# Post-processing of recorded spectra

Load the on/off spectrum pairs recorded by `rtlsdr-recorder`, clean and
downsample them, average them, and export the result to a FITS spectrum.

In [ ]:
from rtlsdr_recorder import frequency_array
from rtlsdr_recorder.analysis import (
    load_spectrum_pairs,
    clean_spectrum,
    downsample_spectrum,
    average_spectra,
    plot_spectrum_pair,
    write_fits,
)

Load all matched on/off spectrum pairs:

In [ ]:
spectra_on, spectra_off = load_spectrum_pairs('raw')
len(spectra_on)

Plot the first pair and its difference:

In [ ]:
plot_spectrum_pair(spectra_on[0], spectra_off[0], lw=0.1);

The spectra have RFI spikes, and there is always a spike at the center of
the band, so mask both with `clean_spectrum` (sigma clipping plus a mask on
the central channels):

In [ ]:
spectra_on = [clean_spectrum(spec) for spec in spectra_on]
spectra_off = [clean_spectrum(spec) for spec in spectra_off]
plot_spectrum_pair(spectra_on[0], spectra_off[0], lw=0.1);

Spectrally downsample the cleaned spectra:

In [ ]:
DOWNSAMPLE = 10
freq = downsample_spectrum(frequency_array(), DOWNSAMPLE)
spectra_on = [downsample_spectrum(spec, DOWNSAMPLE) for spec in spectra_on]
spectra_off = [downsample_spectrum(spec, DOWNSAMPLE) for spec in spectra_off]
plot_spectrum_pair(spectra_on[0], spectra_off[0], frequencies=freq);

That was the first pair only, so now average all pairs:

In [ ]:
average_on = average_spectra(spectra_on)
average_off = average_spectra(spectra_off)
plot_spectrum_pair(average_on, average_off, frequencies=freq);

Export the averaged difference spectrum to a FITS file:

In [ ]:
write_fits(average_on - average_off, 'spectrum.fits', frequencies=freq, overwrite=True)